In [ ]:
import zipfile, json
import pandas as pd
from pathlib import Path

zip_path = Path("/content/congresstweets0617.zip")

all_tweets = []

with zipfile.ZipFile(zip_path, "r") as z:
    for filename in z.namelist():
        if filename.endswith(".json"):
            with z.open(filename) as f:
                tweets = json.load(f)
                for t in tweets:
                    t["date_file"] = filename  # keep track of which day’s file
                    all_tweets.append(t)

df = pd.DataFrame(all_tweets)
print(df.shape)
df.head()


(22325, 8)


,id,screen_name,time,link,text,source,user_id,date_file
0,877383834240667648,KamalaHarris,2017-06-21T00:33:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,No public hearings. \nNo debate. \nNo text.\nN...,Sprout Social,30354991,2017-06-21.json
1,877376225664090112,tedlieu,2017-06-21T00:02:47-04:00,https://www.twitter.com/tedlieu/statuses/87737...,Why should world leaders trust or listen to US...,Twitter for Android,21059255,2017-06-21.json
2,877405817565306880,JerryNadler,2017-06-21T02:00:22-04:00,https://www.twitter.com/JerryNadler/statuses/8...,Congratulations to Jon @ossoff and #GADems on ...,Twitter for iPhone,408816873,2017-06-21.json
3,877400668411052032,TulsiGabbard,2017-06-21T01:39:55-04:00,https://www.twitter.com/TulsiGabbard/statuses/...,Join me in celebrating the gift of yoga on thi...,Twitter for iPhone,26637348,2017-06-21.json
4,877409109846478848,JerryNadler,2017-06-21T02:13:27-04:00,https://www.twitter.com/JerryNadler/statuses/8...,#GA06 is #GOP stronghold &amp; @ossoff nearly ...,Twitter for iPhone,408816873,2017-06-21.json


In [ ]:
with open("/content/historical-users-filtered.json", "r") as f:
  data = json.load(f)
users = pd.json_normalize(
    data,
    record_path=["accounts"],           # path to the nested list
    meta=["name", "chamber", "type", "party"], errors = "ignore",  record_prefix="account_",  # avoid collisions
    meta_prefix="meta_")
users.head()

,account_id,account_screen_name,account_account_type,account_prev_names,account_party,account_deleted,account_name,account_type,account_chamber,meta_name,meta_chamber,meta_type,meta_party
0,899998766845100032,ASEANCaucus,office,NaN,NaN,NaN,NaN,NaN,NaN,ASEAN Caucus,house,caucus,N/A
1,817097052375187457,BlueCollarDems,office,NaN,NaN,NaN,NaN,NaN,NaN,Blue Collar Caucus,house,caucus,D
2,224685124,HouseBlueDogs,office,NaN,NaN,NaN,NaN,NaN,NaN,Blue Dog Coalition,house,caucus,D
3,951207213728698374,AntitrustCaucus,office,NaN,NaN,NaN,NaN,NaN,NaN,Congressional Antitrust Caucus,house,caucus,D
4,192955168,CAPAC,office,NaN,NaN,NaN,NaN,NaN,NaN,Congressional Asian Pacific American Caucus,house,caucus,D


In [ ]:
users.loc[:, "account_prev_names" : "account_chamber"].isna().mean()

,0
account_prev_names,0.919355
account_party,0.954715
account_deleted,0.988834
account_name,0.988834
account_type,0.999380
account_chamber,0.998759


For every single column, over 90% of each one is null, so I'm going to just drop these columns.

In [ ]:
users.loc[:, "account_prev_names":"account_chamber"].

SyntaxError: invalid syntax (ipython-input-1441139792.py, line 1)

In [ ]:
users1 = users[[x for x in users if x not in users.loc[:, "account_prev_names":"account_chamber"].columns]]
#users1 = users.drop(columns = ['col_names'...])
users1

,account_id,account_screen_name,account_account_type,meta_name,meta_chamber,meta_type,meta_party
0,899998766845100032,ASEANCaucus,office,ASEAN Caucus,house,caucus,N/A
1,817097052375187457,BlueCollarDems,office,Blue Collar Caucus,house,caucus,D
2,224685124,HouseBlueDogs,office,Blue Dog Coalition,house,caucus,D
3,951207213728698374,AntitrustCaucus,office,Congressional Antitrust Caucus,house,caucus,D
4,192955168,CAPAC,office,Congressional Asian Pacific American Caucus,house,caucus,D
...,...,...,...,...,...,...,...
1607,1148973240355827713,SenDemsClimate,office,Senate Democrats,senate,party,D
1608,5693842,NRSC,campaign,Senate Republicans,senate,party,R
1609,14344823,SenateGOP,office,Senate Republicans,senate,party,R
1610,114896223,SenateRPC,office,Senate Republicans,senate,party,R


In [ ]:
display(users1.shape)
display(df.shape)


(1612, 7)

(22325, 8)

In [ ]:
len(df["screen_name"].unique())

794

There are more available users than total unqiue tweets in the 2017 06 month tweet dataset. This is good.

In [ ]:
users1[users1["account_screen_name"] == "KamalaHarris"]

,account_id,account_screen_name,account_account_type,meta_name,meta_chamber,meta_type,meta_party
1357,30354991,KamalaHarris,campaign,Kamala Harris,senate,member,D


Doesn't look like the account id given in the users data matches with the account id given in the tweets data.

In [ ]:
df[df["screen_name"] == "KamalaHarris"]

,id,screen_name,time,link,text,source,user_id,date_file
0,877383834240667648,KamalaHarris,2017-06-21T00:33:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,No public hearings. \nNo debate. \nNo text.\nN...,Sprout Social,30354991,2017-06-21.json
569,877553682023317505,KamalaHarris,2017-06-21T11:47:56-04:00,https://www.twitter.com/KamalaHarris/statuses/...,Thank you to all the people who fought hard in...,Sprout Social,30354991,2017-06-21.json
802,877569054030082048,KamalaHarris,2017-06-21T12:49:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,The GOP health care repeal would leave many wh...,Sprout Social,30354991,2017-06-21.json
1271,877599457419837440,KamalaHarris,2017-06-21T14:49:50-04:00,https://www.twitter.com/KamalaHarris/statuses/...,We cannot despair — progress does not happen o...,Sprout Social,30354991,2017-06-21.json
1752,877630207619977217,KamalaHarris,2017-06-21T16:52:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,The GOP wants to vote next week on a health ca...,Sprout Social,30354991,2017-06-21.json
...,...,...,...,...,...,...,...,...
21630,880854202326568960,KamalaHarris,2017-06-30T14:23:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,"No regrets. NONE. Call everyone you know, talk...",Sprout Social,30354991,2017-06-30.json
21735,880875203248754692,KamalaHarris,2017-06-30T15:46:28-04:00,https://www.twitter.com/KamalaHarris/statuses/...,Republicans are actively trying to take away h...,Sprout Social,30354991,2017-06-30.json
21889,880893730487775232,KamalaHarris,2017-06-30T17:00:06-04:00,https://www.twitter.com/KamalaHarris/statuses/...,"This weekend we need to talk, march, call, ema...",Sprout Social,30354991,2017-06-30.json
22250,880926174616145920,KamalaHarris,2017-06-30T19:09:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,We need to stop saying folks are going to lose...,Sprout Social,30354991,2017-06-30.json


Looks like it is a id for each individual tweet and not for each account.

In [ ]:
merged = pd.merge(left = df, right = users1, left_on = "screen_name", right_on = "account_screen_name")

In [ ]:
merged.head()

,id,screen_name,time,link,text,source,user_id,date_file,account_id,account_screen_name,account_account_type,meta_name,meta_chamber,meta_type,meta_party
0,877383834240667648,KamalaHarris,2017-06-21T00:33:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,No public hearings. \nNo debate. \nNo text.\nN...,Sprout Social,30354991,2017-06-21.json,30354991,KamalaHarris,campaign,Kamala Harris,senate,member,D
1,877376225664090112,tedlieu,2017-06-21T00:02:47-04:00,https://www.twitter.com/tedlieu/statuses/87737...,Why should world leaders trust or listen to US...,Twitter for Android,21059255,2017-06-21.json,21059255,tedlieu,campaign,Ted Lieu,house,member,D
2,877405817565306880,JerryNadler,2017-06-21T02:00:22-04:00,https://www.twitter.com/JerryNadler/statuses/8...,Congratulations to Jon @ossoff and #GADems on ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D
3,877400668411052032,TulsiGabbard,2017-06-21T01:39:55-04:00,https://www.twitter.com/TulsiGabbard/statuses/...,Join me in celebrating the gift of yoga on thi...,Twitter for iPhone,26637348,2017-06-21.json,26637348,TulsiGabbard,campaign,Tulsi Gabbard,house,member,D
4,877409109846478848,JerryNadler,2017-06-21T02:13:27-04:00,https://www.twitter.com/JerryNadler/statuses/8...,#GA06 is #GOP stronghold &amp; @ossoff nearly ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D


In [ ]:
merged["meta_type"].unique()

array(['member', 'committee', 'party', 'caucus'], dtype=object)

Columns like source, datefile are probably useless. Will convert time to a normal date time and drop date_file.

In [ ]:
merged["time"] = pd.to_datetime(merged["time"])

In [ ]:
merged.head()

,id,screen_name,time,link,text,source,user_id,date_file,account_id,account_screen_name,account_account_type,meta_name,meta_chamber,meta_type,meta_party
0,877383834240667648,KamalaHarris,2017-06-21 00:33:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,No public hearings. \nNo debate. \nNo text.\nN...,Sprout Social,30354991,2017-06-21.json,30354991,KamalaHarris,campaign,Kamala Harris,senate,member,D
1,877376225664090112,tedlieu,2017-06-21 00:02:47-04:00,https://www.twitter.com/tedlieu/statuses/87737...,Why should world leaders trust or listen to US...,Twitter for Android,21059255,2017-06-21.json,21059255,tedlieu,campaign,Ted Lieu,house,member,D
2,877405817565306880,JerryNadler,2017-06-21 02:00:22-04:00,https://www.twitter.com/JerryNadler/statuses/8...,Congratulations to Jon @ossoff and #GADems on ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D
3,877400668411052032,TulsiGabbard,2017-06-21 01:39:55-04:00,https://www.twitter.com/TulsiGabbard/statuses/...,Join me in celebrating the gift of yoga on thi...,Twitter for iPhone,26637348,2017-06-21.json,26637348,TulsiGabbard,campaign,Tulsi Gabbard,house,member,D
4,877409109846478848,JerryNadler,2017-06-21 02:13:27-04:00,https://www.twitter.com/JerryNadler/statuses/8...,#GA06 is #GOP stronghold &amp; @ossoff nearly ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D


Text Cleaning and Preprocessing


In [ ]:
merged2 = merged.copy()

Convert to lower case

In [ ]:
merged2["Clean_Text"] = merged2["text"].str.lower()


In [ ]:
merged2.head()

,id,screen_name,time,link,text,source,user_id,date_file,account_id,account_screen_name,account_account_type,meta_name,meta_chamber,meta_type,meta_party,Clean_Text
0,877383834240667648,KamalaHarris,2017-06-21 00:33:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,No public hearings. \nNo debate. \nNo text.\nN...,Sprout Social,30354991,2017-06-21.json,30354991,KamalaHarris,campaign,Kamala Harris,senate,member,D,no public hearings. \nno debate. \nno text.\nn...
1,877376225664090112,tedlieu,2017-06-21 00:02:47-04:00,https://www.twitter.com/tedlieu/statuses/87737...,Why should world leaders trust or listen to US...,Twitter for Android,21059255,2017-06-21.json,21059255,tedlieu,campaign,Ted Lieu,house,member,D,why should world leaders trust or listen to us...
2,877405817565306880,JerryNadler,2017-06-21 02:00:22-04:00,https://www.twitter.com/JerryNadler/statuses/8...,Congratulations to Jon @ossoff and #GADems on ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D,congratulations to jon @ossoff and #gadems on ...
3,877400668411052032,TulsiGabbard,2017-06-21 01:39:55-04:00,https://www.twitter.com/TulsiGabbard/statuses/...,Join me in celebrating the gift of yoga on thi...,Twitter for iPhone,26637348,2017-06-21.json,26637348,TulsiGabbard,campaign,Tulsi Gabbard,house,member,D,join me in celebrating the gift of yoga on thi...
4,877409109846478848,JerryNadler,2017-06-21 02:13:27-04:00,https://www.twitter.com/JerryNadler/statuses/8...,#GA06 is #GOP stronghold &amp; @ossoff nearly ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D,#ga06 is #gop stronghold &amp; @ossoff nearly ...


Remove punctuations

In [ ]:
from string import punctuation

In [ ]:
print(type(punctuation))

<class 'str'>


In [ ]:
import re

Also removes new lines characters, replaces important signals like "@" or "#" with a string label.

In [ ]:
def remove_punct(text):
  text = re.sub(r"[^\w\s'@#]", "", text)
  text = re.sub(r"\n", "", text)
  text = re.sub(r"@[\w]+", "<OTHER_USER>", text)
  text = re.sub(r"#", "", text)
  text = re.sub(r"http\S+", "", text)
  text = re.sub(r"[^\x00-\x7F]+", "", text) #this removes non ASCII aka emojies and stuff
  return text

In [ ]:
merged2["Clean_Text"] = merged2["Clean_Text"].apply(remove_punct)

In [ ]:
merged2.head()

,id,screen_name,time,link,text,source,user_id,date_file,account_id,account_screen_name,account_account_type,meta_name,meta_chamber,meta_type,meta_party,Clean_Text
0,877383834240667648,KamalaHarris,2017-06-21 00:33:01-04:00,https://www.twitter.com/KamalaHarris/statuses/...,No public hearings. \nNo debate. \nNo text.\nN...,Sprout Social,30354991,2017-06-21.json,30354991,KamalaHarris,campaign,Kamala Harris,senate,member,D,no public hearings no debate no textno transpa...
1,877376225664090112,tedlieu,2017-06-21 00:02:47-04:00,https://www.twitter.com/tedlieu/statuses/87737...,Why should world leaders trust or listen to US...,Twitter for Android,21059255,2017-06-21.json,21059255,tedlieu,campaign,Ted Lieu,house,member,D,why should world leaders trust or listen to us...
2,877405817565306880,JerryNadler,2017-06-21 02:00:22-04:00,https://www.twitter.com/JerryNadler/statuses/8...,Congratulations to Jon @ossoff and #GADems on ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D,congratulations to jon <OTHER_USER> and gadems...
3,877400668411052032,TulsiGabbard,2017-06-21 01:39:55-04:00,https://www.twitter.com/TulsiGabbard/statuses/...,Join me in celebrating the gift of yoga on thi...,Twitter for iPhone,26637348,2017-06-21.json,26637348,TulsiGabbard,campaign,Tulsi Gabbard,house,member,D,join me in celebrating the gift of yoga on thi...
4,877409109846478848,JerryNadler,2017-06-21 02:13:27-04:00,https://www.twitter.com/JerryNadler/statuses/8...,#GA06 is #GOP stronghold &amp; @ossoff nearly ...,Twitter for iPhone,408816873,2017-06-21.json,408816873,JerryNadler,campaign,Jerrold Nadler,house,member,D,ga06 is gop stronghold amp <OTHER_USER> nearly...


In [ ]:
merged2.loc[0, "Clean_Text"]

'no public hearings no debate no textno transparency on behalf of the american people i demand the gop showusthebill'

In [ ]:
merged2.shape

(20416, 16)

In [ ]:
merged2.to_csv("merged2.csv")

In [ ]:
merged.to_csv("merged.csv")